# Run `954161fb50e6`

**Mode**: `download`
**Tier**: `deep`
**Datasets**: _none selected_

## Request

> Explore why hrvatin and green datasets are not visible to bot users via datasets command.

## Final summary

## Summary
Both datasets are **correctly registered** — all files are present in `/data/green-2023-ppc-inhibitory-snatac/` and `/data/hrvatin-2019-pesca-interneuron/`, and `describe_dataset` returns full, polished cards for each. The problem is **not** missing data or a failed registration. The most likely root cause is a **stale bot registry cache**: `register_dataset` writes both a card file (read live, per-call — which is why cards work) and an entry in `registry.yaml`. Its contract makes datasets live "for all future *jobs*," but the Discord `datasets` listing appears to be served from the bot's in-memory snapshot of `registry.yaml` loaded at startup — before these two were registered (~18:50 UTC 2026‑06‑08) — so it hasn't picked them up.

## What I did
- Listed `/data`: both dataset dirs present with all expected files (green ~9 GB incl. configured `gse232200_snapatac.h5ad`; hrvatin incl. `gse136802_scrna.h5ad` + `gse136802_atac.h5ad`).
- Confirmed `describe_dataset` returns complete cards for both → card half of registration succeeded.
- Read the `register_dataset` schema: it writes a YAML entry **and** a card; "live for future jobs" ≠ the live Discord listing.
- Cross-checked prior runs (registration run 185025 reported "REGISTERED & live").
- Saved a channel note with the diagnosis + recommended fix.

## Caveats / next steps
- **Recommended fix (host-side):** reload/restart the bot's registry cache so the `datasets` command re-reads `registry.yaml`. Re-running `register_dataset` will **not** fix an in-memory cache and risks duplicate entries.
- I could not directly read host-side `registry.yaml` (outside the sandbox) to 100% confirm the YAML entry exists; the inference rests on the prior run returning OK and cards being present.
- A broad `find /` wedged the IPython kernel for the rest of this job, so I couldn't write `/work/datasets_visibility_fix.md` or stage files. With the sandbox back, I'd grep `registry.yaml` to confirm the entries, then advise the maintainer to reload the bot.

In [ ]:
import os, glob
print("=== /data listing ===")
for p in sorted(glob.glob("/data/*")):
    print(p, "DIR" if os.path.isdir(p) else "file")
print("\n=== look for registry files ===")
for pat in ["/data/registry.yaml","/data/registry.yml","/data/*.yaml","/data/*.yml","/data/*.json","/config/*","/work/staged/*"]:
    for p in glob.glob(pat):
        print(p)


exit_code: 0
--- stdout ---
=== /data listing ===
/data/abo-visualcoding-v1 DIR
/data/allen-v1-dev-atlas DIR
/data/bakken2021-biccn-m1 DIR
/data/bugeon2022-v1 DIR
/data/gouwens2020-patchseq DIR
/data/green-2023-ppc-inhibitory-snatac DIR
/data/hrvatin-2019-pesca-interneuron DIR
/data/marmoset-cortical-atlas DIR
/data/microns-functional DIR
/data/microns-minnie65 DIR
/data/neuropixels-visualcoding-v1 DIR
/data/tasic2018-v1 DIR
/data/v1dd DIR
/data/vizgen-merfish-brain-map DIR
/data/yao2023-wmb-10x DIR
/data/zhuang-abca-whole-brain DIR

=== look for registry files ===

--- stderr ---



In [ ]:
import subprocess
print(subprocess.run(["bash","-lc","find / -name 'registry*.y*ml' 2>/dev/null; echo '---'; find / -iname 'registry*' 2>/dev/null | grep -v proc | head -50"],capture_output=True,text=True).stdout)


exit_code: 0
--- stdout ---
---
/usr/local/share/jupyter/lab/schemas/@jupyterlab/toc-extension/registry.json
/usr/local/lib/python3.11/site-packages/numcodecs/registry.py
/usr/local/lib/python3.11/site-packages/numcodecs/__pycache__/registry.cpython-311.pyc
/usr/local/lib/python3.11/site-packages/upath/registry.py
/usr/local/lib/python3.11/site-packages/upath/__pycache__/registry.cpython-311.pyc
/usr/local/lib/python3.11/site-packages/prometheus_client/registry.py
/usr/local/lib/python3.11/site-packages/prometheus_client/__pycache__/registry.cpython-311.pyc
/usr/local/lib/python3.11/site-packages/anndata/_io/specs/registry.py
/usr/local/lib/python3.11/site-packages/anndata/_io/specs/__pycache__/registry.cpython-311.pyc
/usr/local/lib/python3.11/site-packages/tables/registry.py
/usr/local/lib/python3.11/site-packages/tables/__pycache__/registry.cpython-311.pyc
/usr/local/lib/python3.11/site-packages/chardet/registry.py
/usr/local/lib/python3.11/site-packages/chardet/__pycache__/registry

In [ ]:
import os, glob
for d in ["tasic2018-v1","green-2023-ppc-inhibitory-snatac","hrvatin-2019-pesca-interneuron","bakken2021-biccn-m1"]:
    print(f"\n=== /data/{d} ===")
    base=f"/data/{d}"
    for root,dirs,files in os.walk(base):
        for f in sorted(files):
            p=os.path.join(root,f)
            print(f"  {p}  {os.path.getsize(p)/1e6:.1f} MB")


exit_code: 0
--- stdout ---

=== /data/tasic2018-v1 ===
  /data/tasic2018-v1/GSE115746_cells_exon_counts.csv.gz  334.7 MB
  /data/tasic2018-v1/GSE115746_complete_metadata_28706-cells.csv.gz  0.5 MB
  /data/tasic2018-v1/tasic2018_full_counts.h5ad  1352.3 MB
  /data/tasic2018-v1/v1_all_norm.h5ad  1180.1 MB
  /data/tasic2018-v1/v1_all_proc.h5ad  1186.8 MB
  /data/tasic2018-v1/v1_neurons_proc.h5ad  1154.2 MB
  /data/tasic2018-v1/v1_upper_norm.h5ad  265.0 MB
  /data/tasic2018-v1/v1_upper_proc.h5ad  298.2 MB

=== /data/green-2023-ppc-inhibitory-snatac ===
  /data/green-2023-ppc-inhibitory-snatac/GSM7317753_biorep1.snap.gz  2258.5 MB
  /data/green-2023-ppc-inhibitory-snatac/GSM7317753_biorep1_singlecell.csv.gz  5.5 MB
  /data/green-2023-ppc-inhibitory-snatac/GSM7317754_biorep2.snap.gz  2297.0 MB
  /data/green-2023-ppc-inhibitory-snatac/GSM7317754_biorep2_singlecell.csv.gz  5.6 MB
  /data/green-2023-ppc-inhibitory-snatac/gse232200_config.md  0.0 MB
  /data/green-2023-ppc-inhibitory-snatac/gse2

In [ ]:
for f in ["/data/green-2023-ppc-inhibitory-snatac/gse232200_config.md","/data/hrvatin-2019-pesca-interneuron/gse136802_config.md"]:
    print("="*60); print(f); print("="*60)
    print(open(f).read())


exit_code: 0
--- stdout ---
/data/green-2023-ppc-inhibitory-snatac/gse232200_config.md
# GSE232200 — provenance & schema

**Source:** NCBI GEO GSE232200 (Green et al. 2023, Nature, doi:10.1038/s41586-023-06357-1, PMID 37468637).
10x snATAC-seq of Gad2-Cre;Sun1-GFP cortical (PPC) inhibitory neurons. Downloaded 2026-06-08 from
ftp.ncbi.nlm.nih.gov/geo/samples/GSM7317nnn/.

**Raw files committed:** GSM7317753_biorep1.snap.gz, GSM7317754_biorep2.snap.gz (SnapATAC v1, genome mm10),
plus per-biorep CellRanger-ATAC singlecell.csv.gz.

**Configured artifact:** gse232200_snapatac.h5ad — merged peak x cell matrix.
- shape: 22,988 cells x 459,912 peaks; X = peak fragment counts (float32, uint8 origin).
- obs: BD QC fields from .snap (TN,UM,PP,UQ,SE,SA,PE,PL,US,CM), barcode, biorep (biorep1=GSM7317753,
  biorep2=GSM7317754), plus joined CellRanger QC (passed_filters, peak_region_fragments, TSS_fragments,
  promoter/enhancer_region_fragments, is__cell_barcode, cell_id).
- var: chr/start/end (mm10);

In [ ]:
import subprocess
cmds = [
 "ls -la /work/staged 2>/dev/null || echo 'no /work/staged'",
 "find / -path /proc -prune -o -type d -name 'dataset-registry' -print 2>/dev/null",
 "find / -path /proc -prune -o -type d -name 'cards' -print 2>/dev/null | grep -iv site-packages | head",
 "grep -rl 'tasic2018-v1' / 2>/dev/null | grep -v proc | grep -iv site-packages | head",
]
for c in cmds:
    print("$ "+c)
    print(subprocess.run(["bash","-lc",c],capture_output=True,text=True).stdout or "(empty)")
    print("-"*40)


exit_code: 124
--- stdout ---

--- stderr ---
kernel call timed out


In [ ]:
import subprocess, os
for c in [
 "ls -la /work/staged 2>/dev/null || echo 'NO /work/staged'",
 "ls /work 2>/dev/null",
 "ls -d /data/../* 2>/dev/null",
 "find /data -maxdepth 1 -name '*.yaml' -o -maxdepth 1 -name '*.yml' 2>/dev/null",
]:
    print("$ "+c)
    r=subprocess.run(["bash","-lc",c],capture_output=True,text=True,timeout=20)
    print(r.stdout or "(empty)")
    print("-"*30)


exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
import os
print("staged exists:", os.path.exists("/work/staged"))
if os.path.exists("/work/staged"):
    print(os.listdir("/work/staged"))
print("work:", os.listdir("/work")[:50])


exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
print("hello", 1+1)

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
import subprocess; print(subprocess.check_output(['bash','-lc','echo alive; ls /work']).decode('utf-8','replace'))

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
print("ping")

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
2+2

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
print("revive check", 3*3)

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
1

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
import os; print(os.path.exists("/data/green-2023-ppc-inhibitory-snatac"))

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
print("ok")

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
print("ping2")

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
print("ping3")

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
print("ping4", 2**5)

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
print("ping5")

exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
import os
print("alive")
print("staged:", os.path.exists("/work/staged"))
print(os.listdir("/work"))


exit_code: 1
--- stdout ---

--- stderr ---



In [ ]:
print("final-ping")

exit_code: 1
--- stdout ---

--- stderr ---



## Artifacts
- [`.mpl_cache/fontlist-v390.json`](./.mpl_cache/fontlist-v390.json)